# 06 - Temporal, Semantic & Gap Detection

**Objective**: Analyse temporal research trends, compute centroid-debiased semantic similarity between biomedical concepts, calculate document-level co-occurrence, score candidate research gaps, and generate Phase 1 visualizations.

**Inputs**:
- `outputs/mhealth_preprocessed.csv`
- `outputs/document_embeddings.npy`
- `outputs/document_embedding_index.csv`
- `outputs/topic_assignments.csv`
- `outputs/mesh_normalized.csv`
- `outputs/concept_summary.csv`

**Outputs**:
- `outputs/topic_year_matrix.csv`, `outputs/concept_year_matrix.csv`, `outputs/concept_trends.csv`
- `outputs/concept_summary_filtered.csv`, `outputs/concept_quality_filter_report.json`
- `outputs/semantic_similarity_pairs_debiased.csv`, `outputs/concept_cooccurrence.csv`
- `outputs/candidate_research_gaps.csv`, `outputs/gap_evidence.csv`
- 9 presentation figures in `figures/`

In [1]:
import sys
from pathlib import Path

project_root = Path().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

In [2]:
import numpy as np
import pandas as pd
from src.config_loader import set_seed, ensure_dirs

dirs = ensure_dirs()
set_seed()

[CONFIG] Random seed set to 42


## Load Previous Outputs

In [3]:
df = pd.read_csv(dirs['outputs'] / 'mhealth_preprocessed.csv', encoding='utf-8-sig')
assignments = pd.read_csv(dirs['outputs'] / 'topic_assignments.csv', encoding='utf-8-sig')
topic_summary = pd.read_csv(dirs['outputs'] / 'topic_summary.csv', encoding='utf-8-sig')
normalized_concepts = pd.read_csv(dirs['outputs'] / 'mesh_normalized.csv', encoding='utf-8-sig')
concept_summary = pd.read_csv(dirs['outputs'] / 'concept_summary.csv', encoding='utf-8-sig')
embeddings = np.load(dirs['outputs'] / 'document_embeddings.npy')
emb_index = pd.read_csv(dirs['outputs'] / 'document_embedding_index.csv', encoding='utf-8-sig')
print(f'Records: {len(df):,}, Concepts: {len(concept_summary):,}')

Records: 12,642, Concepts: 2,790


## Temporal Trend Analysis

In [4]:
from src.temporal_analysis import (
    build_topic_year_matrix, build_concept_year_matrix,
    analyze_trends, save_temporal_outputs,
)

topic_year_matrix = build_topic_year_matrix(assignments)
concept_year_matrix = build_concept_year_matrix(normalized_concepts, df)
trend_results = analyze_trends(topic_year_matrix, concept_year_matrix)
save_temporal_outputs(topic_year_matrix, concept_year_matrix, trend_results)


[TEMPORAL] Analyzing research trends...



  Topics analyzed: 63
  Concepts analyzed: 300
  Emerging concepts: 50

  Top 10 increasing topics:
    Topic 39: growth=0.274, recent=24
    Topic 38: growth=0.261, recent=31
    Topic 28: growth=0.217, recent=51
    Topic 26: growth=0.211, recent=51
    Topic 6: growth=0.198, recent=114
    Topic 12: growth=0.165, recent=75
    Topic 25: growth=0.164, recent=45
    Topic 17: growth=0.154, recent=69
    Topic 34: growth=0.154, recent=34
    Topic 4: growth=0.150, recent=130

  Top 10 emerging concepts:
    interviews as topic: growth=0.753, recent=68
    stress, psychological: growth=0.656, recent=44
    patient education as topic: growth=0.602, recent=50
    patient acceptance of health care: growth=0.562, recent=112
    middle aged: growth=0.555, recent=1153
    rural health services: growth=0.550, recent=17
    patient preference: growth=0.530, recent=29
    attitude of health personnel: growth=0.524, recent=54
    aged, 80 and over: growth=0.513, recent=126
    male: growth=0.497

{'topic_year_matrix': WindowsPath('E:/Major_Project_Implementation/outputs/topic_year_matrix.csv'),
 'concept_year_matrix': WindowsPath('E:/Major_Project_Implementation/outputs/concept_year_matrix.csv'),
 'topic_trends': WindowsPath('E:/Major_Project_Implementation/outputs/topic_trends.csv'),
 'concept_trends': WindowsPath('E:/Major_Project_Implementation/outputs/concept_trends.csv'),
 'emerging_concepts': WindowsPath('E:/Major_Project_Implementation/outputs/emerging_concepts.csv')}

## Concept Quality Filtering & Centroid Debiasing

Raw concept embeddings suffer from centroid collapse (pairwise similarities ~0.996-1.000).
We apply transparent quality filtering (removing demographics, generic study designs, and broad platform terms)
and centroid debiasing (mean-centering) to restore discriminative semantic similarities.

In [5]:
from src.concept_quality_filter import (
    filter_concepts,
    compute_debiased_concept_embeddings,
    compute_debiased_semantic_similarity,
    save_quality_filter_outputs,
    deduplicate_synonym_pairs,
)
from src.gap_detection import compute_cooccurrence

# 1. Quality filter concepts (min 20 docs, <= 60% corpus, blocklist)
filtered_summary, filtered_doc_concepts, filter_report = filter_concepts(
    concept_summary,
    normalized_concepts,
    concept_col="normalized_concept",
    min_doc_count=20,
    max_doc_fraction=0.60,
)

# 2. Centroid-debiased concept embeddings
concept_embs, corpus_mean = compute_debiased_concept_embeddings(
    filtered_summary,
    filtered_doc_concepts,
    embeddings,
    emb_index,
    concept_col="normalized_concept",
)

# 3. Debiased pairwise semantic similarity
semantic_pairs = compute_debiased_semantic_similarity(
    concept_embs,
    top_n_pairs=5000,
    min_similarity=0.30,
)

# 4. Document-level concept co-occurrence
fd = filtered_doc_concepts.copy()
fd["concept"] = fd["normalized_concept"]
cooccurrence_df = compute_cooccurrence(fd, top_n_concepts=600)

save_quality_filter_outputs(filtered_summary, filter_report, semantic_pairs)


[QUALITY] Concept quality filtering report:
  Total before  : 2,790
  Blocked (heuristic) : 177
  Too rare (<20 docs) : 1,875
  Too common (>60% of corpus): 0
  KEPT          : 738

  Sample blocked concepts:
    - humans
    - mhealth
    - telemedicine
    - female
    - male
    - mobile applications
    - adult
    - mobile health
    - middle aged
    - smartphone
    - aged
    - adolescent
    - young adult
    - digital health
    - mobile phone

[QUALITY] Computing centroid-debiased concept embeddings...
  Corpus centroid norm: 0.9424
  Corpus centroid cos to self: 1.0000  (reference)


  Concepts with embeddings: 737  (skipped: 1)

  Debiased similarity distribution (sample 200 concepts):
    mean=0.0252  std=0.2951  min=-0.7169  max=0.9922

[QUALITY] Computing debiased pairwise similarity for 737 concepts...
  Potential pairs: 271,216
  Pairs above threshold (0.3): 5,000
  Similarity range: 0.634 – 0.996

[GAP] Computing concept co-occurrence...


  Concepts: 600
  Computing pairwise co-occurrence...


Co-occurrence:   0%|          | 0/600 [00:00<?, ?it/s]

Co-occurrence:  10%|█         | 62/600 [00:00<00:00, 619.82it/s]

...

Co-occurrence:  48%|████▊     | 285/600 [00:00<00:00, 691.75it/s]

Co-occurrence:  61%|██████    | 367/600 [00:00<00:00, 727.94it/s]

Co-occurrence: 100%|██████████| 600/600 [00:00<00:00, 978.39it/s]

  Co-occurring pairs (>=2): 17,196
[SAVE] Filtered concept summary -> E:\Major_Project_Implementation\outputs\concept_summary_filtered.csv
[SAVE] Quality filter report -> E:\Major_Project_Implementation\outputs\concept_quality_filter_report.json
[SAVE] Debiased semantic pairs -> E:\Major_Project_Implementation\outputs\semantic_similarity_pairs_debiased.csv


{'filtered_summary': WindowsPath('E:/Major_Project_Implementation/outputs/concept_summary_filtered.csv'),
 'filter_report': WindowsPath('E:/Major_Project_Implementation/outputs/concept_quality_filter_report.json'),
 'debiased_pairs': WindowsPath('E:/Major_Project_Implementation/outputs/semantic_similarity_pairs_debiased.csv')}

## Candidate Research Gap Detection

**GapScore = SemanticSimilarity x (1 - NormalizedCoOccurrence) x (1 + TemporalGrowthAvg)**

In [6]:
from src.gap_detection import detect_candidate_gaps, prepare_gap_evidence, save_gap_outputs

ct_filtered = trend_results["concept_trends"][
    trend_results["concept_trends"]["concept"].isin(filtered_summary["concept"].tolist())
].reset_index(drop=True)

gaps_df = detect_candidate_gaps(
    semantic_pairs=semantic_pairs,
    cooccurrence_df=cooccurrence_df,
    concept_trends=ct_filtered,
    doc_concepts=fd,
    topic_assignments=assignments,
)

evidence_df = prepare_gap_evidence(
    gaps_df, df, ct_filtered, topic_summary,
)

# Flag & deprioritize synonym/variant pairs
gaps_df = deduplicate_synonym_pairs(gaps_df, sim_threshold=0.90)

save_gap_outputs(semantic_pairs, cooccurrence_df, gaps_df, evidence_df)


CANDIDATE RESEARCH GAP DETECTION



  Total semantic pairs evaluated: 5,000
  Candidate research gaps found: 50

  Top 10 candidate research gaps:
    1. [health services accessibility] <-> [rural health services]
       GapScore=1.181  Sim=0.857  CoOc=0.059  Growth=0.463  Joint=15
    2. [anti-hiv agents] <-> [antiretroviral therapy]
       GapScore=1.139  Sim=0.937  CoOc=0.000  Growth=0.216  Joint=8
    3. [mobile health units] <-> [rural health services]
       GapScore=1.128  Sim=0.813  CoOc=0.065  Growth=0.483  Joint=10
    4. [community health workers] <-> [rural health services]
       GapScore=1.119  Sim=0.864  CoOc=0.070  Growth=0.392  Joint=18
    5. [computer security] <-> [security]
       GapScore=1.104  Sim=0.949  CoOc=0.000  Growth=0.163  Joint=8
    6. [reproductive health] <-> [sexual and reproductive health]
       GapScore=1.102  Sim=0.954  CoOc=0.000  Growth=0.155  Joint=12
    7. [anti-hiv agents] <-> [hiv]
       GapScore=1.097  Sim=0.941  CoOc=0.059  Growth=0.239  Joint=22
    8. [stress managemen

{'semantic_pairs': WindowsPath('E:/Major_Project_Implementation/outputs/semantic_similarity_pairs.csv'),
 'cooccurrence': WindowsPath('E:/Major_Project_Implementation/outputs/concept_cooccurrence.csv'),
 'gaps': WindowsPath('E:/Major_Project_Implementation/outputs/candidate_research_gaps.csv'),
 'evidence': WindowsPath('E:/Major_Project_Implementation/outputs/gap_evidence.csv'),
 'quality_audit': WindowsPath('E:/Major_Project_Implementation/outputs/gap_quality_audit.csv')}

## Generate All Figures

In [7]:
from src.visualization import generate_all_figures

fig_data = {
    'df': df,
    'topic_summary': topic_summary,
    'topic_year_matrix': topic_year_matrix,
    'concept_summary': concept_summary,
    'concept_year_matrix': concept_year_matrix,
    'cooccurrence': cooccurrence_df,
    'semantic_pairs': semantic_pairs,
    'gaps': gaps_df,
}
figure_paths = generate_all_figures(fig_data)


GENERATING FIGURES


[FIG] Saved -> E:\Major_Project_Implementation\figures\papers_by_year.png


[FIG] Saved -> E:\Major_Project_Implementation\figures\topic_frequency.png


[FIG] Saved -> E:\Major_Project_Implementation\figures\topic_distribution.png


[FIG] Saved -> E:\Major_Project_Implementation\figures\topic_trends.png


[FIG] Saved -> E:\Major_Project_Implementation\figures\top_concepts.png


[FIG] Saved -> E:\Major_Project_Implementation\figures\concept_trends.png


[FIG] Saved -> E:\Major_Project_Implementation\figures\concept_cooccurrence_heatmap.png


[FIG] Saved -> E:\Major_Project_Implementation\figures\semantic_relationships.png


[FIG] Saved -> E:\Major_Project_Implementation\figures\candidate_gap_ranking.png

[FIG] Generated 9 figures


## Top Candidate Research Gaps

In [8]:
gaps_df[['concept_a', 'concept_b', 'gap_score', 'semantic_similarity', 'cooccurrence_score', 'temporal_growth_avg', 'papers_joint']].head(20)

,concept_a,concept_b,gap_score,semantic_similarity,cooccurrence_score,temporal_growth_avg,papers_joint
0,health services accessibility,rural health services,1.1809,0.8573,0.0586,0.4633,15
1,mobile health units,rural health services,1.1276,0.8126,0.0645,0.4834,10
2,community health workers,rural health services,1.1189,0.8641,0.0700,0.3924,18
3,anti-hiv agents,hiv,1.0975,0.9413,0.0590,0.2390,22
4,health services accessibility,mobile health units,1.0910,0.8716,0.1037,0.3966,28
5,health facilities,rural health services,1.0892,0.8543,0.0000,0.2750,0
6,community health services,rural health services,1.0805,0.8752,0.0317,0.2750,4
7,maternal health services,rural health services,1.0797,0.8174,0.0500,0.3904,7
8,mental disorders,mental health services,1.0778,0.9102,0.1151,0.3382,16
9,anti-hiv agents,hiv testing,1.0558,0.8683,0.0000,0.2159,0


## Phase 1 Complete

Phase 2 will consume:
- `outputs/candidate_research_gaps.csv`
- `outputs/gap_evidence.csv`